# Phase 8 Wave 1 — Model-Family Comparison

This notebook executes the authorized Phase 8 Wave 1 comparison from the committed planning package.

Hard boundaries:

- Wave 1 sklearn-native models only.
- Fixed F2 feature set from Phase 7/7B.
- Frozen Phase 6 folds; folds are loaded, not recomputed.
- ROC-AUC uses verified positive-class probabilities for `Drafted = 1`.
- No HPO, submissions, ensembles, leaderboard use, external data, package installation, or environment mutation.
- No Phase 9, Phase 10, or Phase 11 work.

## 1. Setup, Imports, and Constants

**Purpose.** Record the authorized hash, environment expectations, model registry boundaries, and artifact paths.

**Expected output.** Constants are defined; imports succeed only for the authorized sklearn-native stack.

**Leakage caution.** No data is loaded or transformed in this cell.

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import platform
import subprocess
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

AUTHORIZED_HEAD = "f8c791103adc4a259e68a9fff2ccd3e43166801c"
EXPERIMENT_ID = "phase8_model_family_comparison_v1"
PROJECT_SEED = 42

EXPECTED_TRAIN_ROWS = 2781
EXPECTED_TEST_ROWS = 696
EXPECTED_FOLD_SHA16 = "96937649526bcadb"
EXPECTED_F2_AUC = 0.8116502602456482
M0_TOLERANCE = 1e-6
OOF_GAIN_THRESHOLD = 0.005436
SAME_SIGN_FOLD_THRESHOLD = 4
MIN_SLICE_SIZE = 50
SLICE_DEGRADATION_THRESHOLD = 0.02

BASE_FEATURES = [
    "Year",
    "Age",
    "Height",
    "Weight",
    "Sprint_40yd",
    "Vertical_Jump",
    "Bench_Press_Reps",
    "Broad_Jump",
    "Agility_3cone",
    "Shuttle",
    "Player_Type",
    "Position_Type",
    "Position",
]
CATEGORICAL_FEATURES = ["Player_Type", "Position_Type", "Position"]
PHYSICAL_TEST_COLUMNS = [
    "Sprint_40yd",
    "Vertical_Jump",
    "Bench_Press_Reps",
    "Broad_Jump",
    "Agility_3cone",
    "Shuttle",
]
MISSINGNESS_FLAGS = [
    "Age_missing",
    "Sprint_40yd_missing",
    "Vertical_Jump_missing",
    "Bench_Press_Reps_missing",
    "Broad_Jump_missing",
    "Agility_3cone_missing",
    "Shuttle_missing",
]
EXPECTED_MISSING_COUNTS = {
    "Age": 435,
    "Sprint_40yd": 145,
    "Vertical_Jump": 554,
    "Bench_Press_Reps": 721,
    "Broad_Jump": 581,
    "Agility_3cone": 970,
    "Shuttle": 912,
}
F2_FEATURES = BASE_FEATURES + MISSINGNESS_FLAGS + ["available_measurement_count"]
FORBIDDEN_FEATURES = {"Id", "Drafted", "School", "measurement_completeness_group", "frequent_vs_rare_school_group"}

MODEL_KEYS = [
    "m0_random_forest_frozen",
    "m1_logistic_regression",
    "m2_random_forest_default",
    "m3_extra_trees_default",
    "m4_hist_gradient_boosting",
]

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / ".git").exists() and (candidate / "data" / "input" / "train.csv").exists():
            return candidate
    raise RuntimeError("Could not locate repository root")

ROOT = find_project_root(Path.cwd())

PATHS = {
    "train": ROOT / "data" / "input" / "train.csv",
    "test": ROOT / "data" / "input" / "test.csv",
    "sample_submission": ROOT / "data" / "input" / "sample_submission.csv",
    "folds": ROOT / "outputs" / "folds" / "phase6_rf_sanity_baseline_v1_fold_assignments.csv",
    "f2_oof": ROOT / "outputs" / "oof" / "phase7_missingness_availability_v1_phase7_f2_median_flags_count_oof_predictions.csv",
    "main_log": ROOT / "logs" / "experiment_log.csv",
    "model_summary": ROOT / "outputs" / "validation" / f"{EXPERIMENT_ID}_model_summary.csv",
    "fold_metrics": ROOT / "outputs" / "validation" / f"{EXPERIMENT_ID}_fold_metrics.csv",
    "slice_report": ROOT / "outputs" / "validation" / f"{EXPERIMENT_ID}_slice_report.csv",
    "validation_report": ROOT / "outputs" / "reports" / f"{EXPERIMENT_ID}_validation_report.md",
    "candidate_log": ROOT / "outputs" / "reports" / f"{EXPERIMENT_ID}_experiment_log_candidate.csv",
    "artifact_manifest": ROOT / "outputs" / "reports" / f"{EXPERIMENT_ID}_artifact_manifest.csv",
}
OOF_PATHS = {
    model_key: ROOT / "outputs" / "oof" / f"{EXPERIMENT_ID}_{model_key}_oof_predictions.csv"
    for model_key in MODEL_KEYS
}

print(f"Repository root: {ROOT}")
print(f"Experiment: {EXPERIMENT_ID}")

Repository root: C:\GitHub\reto_Tokio
Experiment: phase8_model_family_comparison_v1


## 2. Repository, Environment, and Artifact Gates

**Purpose.** Enforce the Project Authorization Note before any computation.

**Expected output.** HEAD, environment versions, staged files, forbidden diffs, external GBDT absence, and artifact collisions are checked.

**Leakage caution.** This is an operational gate only. It does not read labels except through later contract checks.

In [2]:
def git_stdout(args: list[str]) -> str:
    result = subprocess.run(["git", *args], cwd=ROOT, text=True, capture_output=True, check=True)
    return result.stdout.strip()

full_head = git_stdout(["rev-parse", "HEAD"])
short_head = git_stdout(["rev-parse", "--short", "HEAD"])
if full_head != AUTHORIZED_HEAD:
    raise RuntimeError(f"HEAD mismatch: expected {AUTHORIZED_HEAD}, got {full_head}")

diff_check = subprocess.run(["git", "diff", "--check"], cwd=ROOT, text=True, capture_output=True)
if diff_check.returncode != 0:
    raise RuntimeError(f"git diff --check failed:\n{diff_check.stdout}\n{diff_check.stderr}")

staged_files = git_stdout(["diff", "--cached", "--name-only"])
if staged_files:
    raise RuntimeError(f"Unexpected staged files:\n{staged_files}")

forbidden_diff = git_stdout([
    "diff",
    "--name-only",
    "--",
    "data/input",
    "notebooks/_official",
    "references",
    "outputs/submissions",
    "logs/experiment_log.csv",
    ".vscode/settings.json",
])
if forbidden_diff:
    raise RuntimeError(f"Forbidden-path diff detected:\n{forbidden_diff}")

versions = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "numpy": np.__version__,
}
expected_versions = {
    "python": "3.13.13",
    "pandas": "3.0.3",
    "sklearn": "1.9.0",
    "numpy": "2.4.6",
}
if versions != expected_versions:
    raise RuntimeError(f"Environment mismatch: expected {expected_versions}, got {versions}")

import importlib.util
external_gbdts = [name for name in ("xgboost", "lightgbm", "catboost") if importlib.util.find_spec(name)]
if external_gbdts:
    raise RuntimeError(f"External GBDT libraries present in Wave 1 environment: {external_gbdts}")

all_output_paths = [*OOF_PATHS.values(), PATHS["model_summary"], PATHS["fold_metrics"], PATHS["slice_report"], PATHS["validation_report"], PATHS["candidate_log"], PATHS["artifact_manifest"]]
collisions = [str(path.relative_to(ROOT)) for path in all_output_paths if path.exists()]
if collisions:
    raise RuntimeError(f"Phase 8 artifact collision; refusing overwrite:\n" + "\n".join(collisions))

main_log_before = PATHS["main_log"].read_bytes() if PATHS["main_log"].exists() else b""

gate_record = {
    "full_head": full_head,
    "short_head": short_head,
    "git_diff_check": "pass",
    "staged_files": staged_files or "",
    "forbidden_diff": forbidden_diff or "",
    "versions": versions,
    "external_gbdts_present": external_gbdts,
    "artifact_collisions": collisions,
}
gate_record

{'full_head': 'f8c791103adc4a259e68a9fff2ccd3e43166801c',
 'short_head': 'f8c7911',
 'git_diff_check': 'pass',
 'staged_files': '',
 'forbidden_diff': '',
 'versions': {'python': '3.13.13',
  'pandas': '3.0.3',
  'sklearn': '1.9.0',
  'numpy': '2.4.6'},
 'external_gbdts_present': [],
 'artifact_collisions': []}

## 3. Data, Fold, and Reference Contract

**Purpose.** Load official CSVs, frozen folds, and the accepted Phase 7 F2 OOF reference.

**Expected output.** Contract checks pass; fold SHA/order/labels are valid; F2 OOF AUC recomputes to the accepted reference.

**Leakage caution.** Test data is used only for structural checks. No fitting, preprocessing, selection, or inference touches test rows.

In [3]:
for required_name in ("train", "test", "sample_submission", "folds", "f2_oof"):
    if not PATHS[required_name].exists():
        raise RuntimeError(f"Missing required input artifact: {PATHS[required_name].relative_to(ROOT)}")

train = pd.read_csv(PATHS["train"])
test = pd.read_csv(PATHS["test"])
sample_submission = pd.read_csv(PATHS["sample_submission"])
folds = pd.read_csv(PATHS["folds"])
f2_oof = pd.read_csv(PATHS["f2_oof"])

if train.shape != (EXPECTED_TRAIN_ROWS, 16):
    raise RuntimeError(f"Unexpected train shape: {train.shape}")
if test.shape[0] != EXPECTED_TEST_ROWS:
    raise RuntimeError(f"Unexpected test row count: {test.shape[0]}")
if sample_submission.shape[0] != EXPECTED_TEST_ROWS or list(sample_submission.columns) != ["Id", "Drafted"]:
    raise RuntimeError(f"Unexpected sample submission contract: shape={sample_submission.shape}, columns={sample_submission.columns.tolist()}")
if "Drafted" not in train.columns or "Drafted" in test.columns:
    raise RuntimeError("Target contract failed")
if "Id" not in train.columns or "Id" not in test.columns:
    raise RuntimeError("Id contract failed")
if sorted(train["Drafted"].dropna().unique().tolist()) != [0, 1]:
    raise RuntimeError("Drafted must be binary with labels 0 and 1")

fold_sha16 = hashlib.sha256(PATHS["folds"].read_bytes()).hexdigest()[:16]
if fold_sha16 != EXPECTED_FOLD_SHA16:
    raise RuntimeError(f"Frozen fold SHA mismatch: expected {EXPECTED_FOLD_SHA16}, got {fold_sha16}")
if len(folds) != EXPECTED_TRAIN_ROWS or sorted(folds["fold"].unique().tolist()) != [0, 1, 2, 3, 4]:
    raise RuntimeError("Frozen fold row count or labels invalid")
if not folds["Id"].equals(train["Id"]):
    raise RuntimeError("Frozen fold Id order does not align with train Id order")

required_oof_cols = {"Id", "fold", "y_true", "y_pred_proba"}
if not required_oof_cols.issubset(f2_oof.columns):
    raise RuntimeError(f"F2 OOF missing required columns: {required_oof_cols - set(f2_oof.columns)}")
if len(f2_oof) != EXPECTED_TRAIN_ROWS:
    raise RuntimeError(f"F2 OOF row count mismatch: {len(f2_oof)}")
if not f2_oof["Id"].equals(train["Id"]):
    raise RuntimeError("F2 OOF Id order does not align with train")
if not f2_oof["fold"].equals(folds["fold"]):
    raise RuntimeError("F2 OOF fold assignment does not match frozen folds")
if not np.isfinite(f2_oof["y_pred_proba"]).all() or not f2_oof["y_pred_proba"].between(0, 1).all():
    raise RuntimeError("F2 OOF probabilities invalid")
f2_auc = float(roc_auc_score(f2_oof["y_true"], f2_oof["y_pred_proba"]))
if abs(f2_auc - EXPECTED_F2_AUC) > 1e-12:
    raise RuntimeError(f"F2 OOF AUC mismatch: expected {EXPECTED_F2_AUC}, got {f2_auc}")

contract_record = {
    "train_shape": train.shape,
    "test_shape": test.shape,
    "sample_shape": sample_submission.shape,
    "fold_sha16": fold_sha16,
    "f2_auc": f2_auc,
}
contract_record

{'train_shape': (2781, 16),
 'test_shape': (696, 15),
 'sample_shape': (696, 2),
 'fold_sha16': '96937649526bcadb',
 'f2_auc': 0.8116502602456482}

## 4. F2 Feature Builder and Diagnostics

**Purpose.** Recreate the accepted F2 feature block using the audited Phase 7 row-wise design.

**Expected output.** The 21 F2 features are built; known missing-count asserts pass; diagnostic slice columns are available but excluded from model features.

**Leakage caution.** Missingness flags and counts are row-wise, parameter-free transformations. School appears only in a diagnostic slice and never in the feature matrix.

In [4]:
def add_f2_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["Age_missing"] = out["Age"].isna().astype(int)
    for column in PHYSICAL_TEST_COLUMNS:
        out[f"{column}_missing"] = out[column].isna().astype(int)
    out["available_measurement_count"] = out[PHYSICAL_TEST_COLUMNS].notna().sum(axis=1).astype(int)
    count = out["available_measurement_count"]
    out["measurement_completeness_group"] = np.select(
        [count.eq(0), count.between(1, 3), count.between(4, 5), count.eq(6)],
        [0, 1, 2, 3],
        default=-1,
    ).astype(int)
    return out

train_fe = add_f2_features(train)

missing_base = [column for column in BASE_FEATURES if column not in train_fe.columns]
if missing_base:
    raise RuntimeError(f"Base feature(s) missing from train: {missing_base}")

for column, expected_missing in EXPECTED_MISSING_COUNTS.items():
    observed = int(train_fe[column].isna().sum())
    if observed != expected_missing:
        raise RuntimeError(f"Missing-count assert failed for {column}: expected {expected_missing}, observed {observed}")

flag_to_source = {"Age_missing": "Age", **{f"{column}_missing": column for column in PHYSICAL_TEST_COLUMNS}}
for flag, source in flag_to_source.items():
    if int(train_fe[flag].sum()) != int(train_fe[source].isna().sum()):
        raise RuntimeError(f"Flag-source mismatch for {flag}")
if not train_fe["available_measurement_count"].between(0, 6).all():
    raise RuntimeError("available_measurement_count outside 0..6")
if not train_fe["measurement_completeness_group"].between(0, 3).all():
    raise RuntimeError("measurement_completeness_group outside 0..3")

if FORBIDDEN_FEATURES.intersection(F2_FEATURES):
    raise RuntimeError(f"Forbidden feature in F2 matrix: {FORBIDDEN_FEATURES.intersection(F2_FEATURES)}")

rare_school_threshold = 5
school_counts = train_fe["School"].value_counts(dropna=False)
train_fe["frequent_vs_rare_school_group"] = train_fe["School"].map(school_counts).apply(
    lambda n: "rare" if int(n) < rare_school_threshold else "frequent"
)

X = train_fe[F2_FEATURES].copy()
y = train_fe["Drafted"].astype(int).copy()
fold_array = folds["fold"].to_numpy()

numeric_features = [column for column in F2_FEATURES if column not in CATEGORICAL_FEATURES]
categorical_features = [column for column in CATEGORICAL_FEATURES if column in F2_FEATURES]

feature_record = {
    "feature_count": len(F2_FEATURES),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "missing_flag_sums": {flag: int(train_fe[flag].sum()) for flag in MISSINGNESS_FLAGS},
    "available_measurement_count_counts": train_fe["available_measurement_count"].value_counts().sort_index().to_dict(),
    "measurement_completeness_group_counts": train_fe["measurement_completeness_group"].value_counts().sort_index().to_dict(),
    "school_group_counts": train_fe["frequent_vs_rare_school_group"].value_counts().to_dict(),
}
feature_record

{'feature_count': 21,
 'numeric_features': ['Year',
  'Age',
  'Height',
  'Weight',
  'Sprint_40yd',
  'Vertical_Jump',
  'Bench_Press_Reps',
  'Broad_Jump',
  'Agility_3cone',
  'Shuttle',
  'Age_missing',
  'Sprint_40yd_missing',
  'Vertical_Jump_missing',
  'Bench_Press_Reps_missing',
  'Broad_Jump_missing',
  'Agility_3cone_missing',
  'Shuttle_missing',
  'available_measurement_count'],
 'categorical_features': ['Player_Type', 'Position_Type', 'Position'],
 'missing_flag_sums': {'Age_missing': 435,
  'Sprint_40yd_missing': 145,
  'Vertical_Jump_missing': 554,
  'Bench_Press_Reps_missing': 721,
  'Broad_Jump_missing': 581,
  'Agility_3cone_missing': 970,
  'Shuttle_missing': 912},
 'available_measurement_count_counts': {0: 56,
  1: 240,
  2: 236,
  3: 137,
  4: 269,
  5: 454,
  6: 1389},
 'measurement_completeness_group_counts': {0: 56, 1: 613, 2: 723, 3: 1389},
 'school_group_counts': {'frequent': 2559, 'rare': 222}}

## 5. Fold-Safe Pipelines and Model Registry

**Purpose.** Define exactly the five authorized Wave 1 model configurations and shared fold-safe preprocessing.

**Expected output.** The registry contains only m0-m4; m5 and external GBDTs are absent.

**Leakage caution.** All learned imputers, encoders, and scalers are fitted inside each training fold through sklearn pipelines.

In [5]:
@dataclass(frozen=True)
class ModelSpec:
    model_key: str
    display_name: str
    estimator: object
    add_numeric_scaler: bool = False


def make_one_hot_encoder() -> OneHotEncoder:
    return OneHotEncoder(handle_unknown="ignore", sparse_output=False)


def make_preprocessor(add_numeric_scaler: bool = False) -> ColumnTransformer:
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if add_numeric_scaler:
        numeric_steps.append(("scaler", StandardScaler()))
    numeric_pipe = Pipeline(numeric_steps)
    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", make_one_hot_encoder()),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipe, numeric_features),
            ("categorical", categorical_pipe, categorical_features),
        ],
        remainder="drop",
    )


def model_registry() -> list[ModelSpec]:
    return [
        ModelSpec(
            "m0_random_forest_frozen",
            "RandomForest frozen Phase 7 F2 reference",
            RandomForestClassifier(n_estimators=100, max_depth=5, random_state=PROJECT_SEED, n_jobs=-1),
            add_numeric_scaler=False,
        ),
        ModelSpec(
            "m1_logistic_regression",
            "LogisticRegression",
            LogisticRegression(max_iter=1000, random_state=PROJECT_SEED),
            add_numeric_scaler=True,
        ),
        ModelSpec(
            "m2_random_forest_default",
            "RandomForest default depth",
            RandomForestClassifier(n_estimators=100, max_depth=None, random_state=PROJECT_SEED, n_jobs=-1),
            add_numeric_scaler=False,
        ),
        ModelSpec(
            "m3_extra_trees_default",
            "ExtraTrees default",
            ExtraTreesClassifier(n_estimators=100, random_state=PROJECT_SEED, n_jobs=-1),
            add_numeric_scaler=False,
        ),
        ModelSpec(
            "m4_hist_gradient_boosting",
            "HistGradientBoosting",
            HistGradientBoostingClassifier(random_state=PROJECT_SEED, early_stopping=False),
            add_numeric_scaler=False,
        ),
    ]


registry = model_registry()
registry_keys = [spec.model_key for spec in registry]
if registry_keys != MODEL_KEYS:
    raise RuntimeError(f"Registry mismatch: expected {MODEL_KEYS}, got {registry_keys}")
if any("m5" in key for key in registry_keys):
    raise RuntimeError("m5 is not authorized in this Wave 1 execution")

[(spec.model_key, spec.estimator.__class__.__name__, spec.add_numeric_scaler) for spec in registry]

[('m0_random_forest_frozen', 'RandomForestClassifier', False),
 ('m1_logistic_regression', 'LogisticRegression', True),
 ('m2_random_forest_default', 'RandomForestClassifier', False),
 ('m3_extra_trees_default', 'ExtraTreesClassifier', False),
 ('m4_hist_gradient_boosting', 'HistGradientBoostingClassifier', False)]

## 6. Metric Helpers and Training Loop

**Purpose.** Train each authorized model on frozen folds and produce OOF probabilities.

**Expected output.** One OOF dataframe and fold-metric table per model; m0 reproduces the F2 reference within tolerance.

**Leakage caution.** Validation rows are transformed only through fold-fitted pipeline state. Positive-class probability extraction verifies `classes_`.

In [6]:
def validate_probabilities(proba: np.ndarray, label: str) -> None:
    if len(proba) == 0:
        raise RuntimeError(f"{label}: empty probability vector")
    if not np.isfinite(proba).all():
        raise RuntimeError(f"{label}: probability vector contains non-finite values")
    if np.nanmin(proba) < 0 or np.nanmax(proba) > 1:
        raise RuntimeError(f"{label}: probability vector outside [0, 1]")


def positive_class_proba(fitted_pipeline: Pipeline, X_valid: pd.DataFrame) -> np.ndarray:
    model = fitted_pipeline.named_steps["model"]
    classes = np.asarray(getattr(model, "classes_", []))
    matches = np.where(classes == 1)[0]
    if len(matches) != 1:
        raise RuntimeError(f"Expected label 1 exactly once in classes_, got {classes.tolist()}")
    proba = fitted_pipeline.predict_proba(X_valid)[:, int(matches[0])]
    validate_probabilities(proba, "positive_class_proba")
    return proba


def compute_fold_metrics(model_key: str, oof_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for fold in sorted(oof_df["fold"].unique()):
        group = oof_df[oof_df["fold"] == fold]
        n_positive = int(group["y_true"].sum())
        n_negative = int(len(group) - n_positive)
        if n_positive == 0 or n_negative == 0:
            raise RuntimeError(f"{model_key}: fold {fold} has one class only")
        rows.append(
            {
                "experiment_id": EXPERIMENT_ID,
                "model_key": model_key,
                "fold": int(fold),
                "n_valid": int(len(group)),
                "n_positive": n_positive,
                "n_negative": n_negative,
                "positive_rate": float(group["y_true"].mean()),
                "roc_auc": float(roc_auc_score(group["y_true"], group["y_pred_proba"])),
            }
        )
    return pd.DataFrame(rows)


def run_model(spec: ModelSpec) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    print(f"Training {spec.model_key}...")
    oof_pred = np.full(len(X), np.nan, dtype=float)
    for fold in range(5):
        train_mask = fold_array != fold
        valid_mask = fold_array == fold
        y_train = y.loc[train_mask]
        y_valid = y.loc[valid_mask]
        if y_train.nunique() != 2 or y_valid.nunique() != 2:
            raise RuntimeError(f"{spec.model_key}: fold {fold} has a one-class train or validation split")
        pipeline = Pipeline(
            steps=[
                ("preprocessor", make_preprocessor(add_numeric_scaler=spec.add_numeric_scaler)),
                ("model", spec.estimator),
            ]
        )
        pipeline.fit(X.loc[train_mask], y_train)
        oof_pred[valid_mask] = positive_class_proba(pipeline, X.loc[valid_mask])
    validate_probabilities(oof_pred, f"{spec.model_key}_oof")
    if np.isnan(oof_pred).any():
        raise RuntimeError(f"{spec.model_key}: OOF contains NaN")
    oof_df = pd.DataFrame(
        {
            "Id": train_fe["Id"].to_numpy(),
            "fold": fold_array,
            "y_true": y.to_numpy(),
            "y_pred_proba": oof_pred,
        }
    )
    if len(oof_df) != EXPECTED_TRAIN_ROWS:
        raise RuntimeError(f"{spec.model_key}: OOF row count mismatch")
    if not oof_df["fold"].equals(folds["fold"]):
        raise RuntimeError(f"{spec.model_key}: OOF fold mismatch")
    fold_df = compute_fold_metrics(spec.model_key, oof_df)
    oof_auc = float(roc_auc_score(oof_df["y_true"], oof_df["y_pred_proba"]))
    summary = {
        "experiment_id": EXPERIMENT_ID,
        "model_key": spec.model_key,
        "display_name": spec.display_name,
        "run_status": "trained",
        "model_family": spec.estimator.__class__.__name__,
        "model_params_summary": json.dumps(spec.estimator.get_params(deep=False), sort_keys=True, default=str),
        "feature_set": "Phase 7 F2",
        "feature_count": len(F2_FEATURES),
        "oof_auc": oof_auc,
        "fold_mean_auc": float(fold_df["roc_auc"].mean()),
        "fold_std_auc": float(fold_df["roc_auc"].std(ddof=1)),
        "fold_auc_scores": "|".join(f"{v:.12f}" for v in fold_df["roc_auc"]),
        "hpo_status": "not_run",
        "submission_created": False,
        "leakage_warning": False,
    }
    return oof_df, fold_df, summary


oof_by_model: dict[str, pd.DataFrame] = {}
fold_metrics_parts = []
summary_rows = []

for spec in registry:
    try:
        oof_df, fold_df, summary = run_model(spec)
    except Exception as exc:
        if spec.model_key == "m0_random_forest_frozen":
            raise
        summary = {
            "experiment_id": EXPERIMENT_ID,
            "model_key": spec.model_key,
            "display_name": spec.display_name,
            "run_status": "failed_run",
            "model_family": spec.estimator.__class__.__name__,
            "model_params_summary": json.dumps(spec.estimator.get_params(deep=False), sort_keys=True, default=str),
            "feature_set": "Phase 7 F2",
            "feature_count": len(F2_FEATURES),
            "oof_auc": np.nan,
            "fold_mean_auc": np.nan,
            "fold_std_auc": np.nan,
            "fold_auc_scores": "",
            "hpo_status": "not_run",
            "submission_created": False,
            "leakage_warning": False,
            "failure_reason": repr(exc),
        }
        summary_rows.append(summary)
        continue
    oof_by_model[spec.model_key] = oof_df
    fold_metrics_parts.append(fold_df)
    summary_rows.append(summary)

if "m0_random_forest_frozen" not in oof_by_model:
    raise RuntimeError("m0 did not produce OOF predictions")

m0_auc = float(roc_auc_score(oof_by_model["m0_random_forest_frozen"]["y_true"], oof_by_model["m0_random_forest_frozen"]["y_pred_proba"]))
if abs(m0_auc - EXPECTED_F2_AUC) > M0_TOLERANCE:
    raise RuntimeError(f"M0 integrity gate failed: expected {EXPECTED_F2_AUC} +/- {M0_TOLERANCE}, got {m0_auc}")
m0_max_abs_diff_vs_f2 = float(np.max(np.abs(oof_by_model["m0_random_forest_frozen"]["y_pred_proba"].to_numpy() - f2_oof["y_pred_proba"].to_numpy())))

fold_metrics = pd.concat(fold_metrics_parts, ignore_index=True)
m0_fold_auc = fold_metrics[fold_metrics["model_key"] == "m0_random_forest_frozen"].set_index("fold")["roc_auc"].to_dict()
for idx, row in fold_metrics.iterrows():
    fold_metrics.loc[idx, "m0_fold_auc"] = m0_fold_auc[int(row["fold"])]
    fold_metrics.loc[idx, "delta_vs_m0_fold_auc"] = float(row["roc_auc"] - m0_fold_auc[int(row["fold"])])

pd.DataFrame(summary_rows)

Training m0_random_forest_frozen...


Training m1_logistic_regression...


Training m2_random_forest_default...


Training m3_extra_trees_default...


Training m4_hist_gradient_boosting...


,experiment_id,model_key,display_name,run_status,model_family,model_params_summary,feature_set,feature_count,oof_auc,fold_mean_auc,fold_std_auc,fold_auc_scores,hpo_status,submission_created,leakage_warning
0,phase8_model_family_comparison_v1,m0_random_forest_frozen,RandomForest frozen Phase 7 F2 reference,trained,RandomForestClassifier,"{""bootstrap"": true, ""ccp_alpha"": 0.0, ""class_w...",Phase 7 F2,21,0.811650,0.812456,0.029238,0.789388885748|0.832381561190|0.825854108957|0...,not_run,False,False
1,phase8_model_family_comparison_v1,m1_logistic_regression,LogisticRegression,trained,LogisticRegression,"{""C"": 1.0, ""class_weight"": null, ""dual"": false...",Phase 7 F2,21,0.827082,0.827682,0.029565,0.797529538131|0.850216634704|0.862816961432|0...,not_run,False,False
2,phase8_model_family_comparison_v1,m2_random_forest_default,RandomForest default depth,trained,RandomForestClassifier,"{""bootstrap"": true, ""ccp_alpha"": 0.0, ""class_w...",Phase 7 F2,21,0.805524,0.806100,0.029678,0.790378201142|0.841579657646|0.822274309255|0...,not_run,False,False
3,phase8_model_family_comparison_v1,m3_extra_trees_default,ExtraTrees default,trained,ExtraTreesClassifier,"{""bootstrap"": false, ""ccp_alpha"": 0.0, ""class_...",Phase 7 F2,21,0.789694,0.790354,0.030487,0.754621516197|0.826862703317|0.811826124014|0...,not_run,False,False
4,phase8_model_family_comparison_v1,m4_hist_gradient_boosting,HistGradientBoosting,trained,HistGradientBoostingClassifier,"{""categorical_features"": ""from_dtype"", ""class_...",Phase 7 F2,21,0.809329,0.809203,0.027170,0.798843914297|0.825257475673|0.818879181760|0...,not_run,False,False


## 7. Slice Diagnostics and Classification

**Purpose.** Apply the pre-registered evidence flags: OOF gain, same-sign fold count, and mandatory slice guard versus m0.

**Expected output.** Model summary, fold metrics, and slice diagnostics are fully populated.

**Leakage caution.** Slice variables are diagnostic only; they do not enter any model feature matrix.

In [7]:
SLICE_COLUMNS = [
    "Player_Type",
    "Position_Type",
    "Year",
    "Age_missing",
    "available_measurement_count",
    "measurement_completeness_group",
    "frequent_vs_rare_school_group",
]


def slice_rows_for_model(model_key: str, oof_df: pd.DataFrame) -> list[dict]:
    rows: list[dict] = []
    enriched = oof_df.join(train_fe[SLICE_COLUMNS])
    for slice_name in SLICE_COLUMNS:
        for slice_value, group in enriched.groupby(slice_name, dropna=False, sort=True):
            n = int(len(group))
            n_positive = int(group["y_true"].sum())
            n_negative = int(n - n_positive)
            status = "computed"
            reason = ""
            auc = np.nan
            if n < MIN_SLICE_SIZE:
                status = "skipped_too_small"
                reason = f"n<{MIN_SLICE_SIZE}"
            elif n_positive == 0 or n_negative == 0:
                status = "skipped_one_class"
                reason = "one class only"
            else:
                auc = float(roc_auc_score(group["y_true"], group["y_pred_proba"]))
            rows.append(
                {
                    "experiment_id": EXPERIMENT_ID,
                    "model_key": model_key,
                    "slice_name": slice_name,
                    "slice_value": str(slice_value),
                    "n": n,
                    "n_positive": n_positive,
                    "n_negative": n_negative,
                    "positive_rate": float(group["y_true"].mean()) if n else np.nan,
                    "roc_auc": auc,
                    "status": status,
                    "reason_if_skipped": reason,
                    "notes": "diagnostic_only" if slice_name in {"frequent_vs_rare_school_group", "measurement_completeness_group"} else "",
                }
            )
    return rows


slice_report = pd.DataFrame(
    [row for model_key, oof_df in oof_by_model.items() for row in slice_rows_for_model(model_key, oof_df)]
)

m0_slice = slice_report[slice_report["model_key"] == "m0_random_forest_frozen"][
    ["slice_name", "slice_value", "roc_auc", "status"]
].rename(columns={"roc_auc": "m0_slice_auc", "status": "m0_slice_status"})
slice_report = slice_report.merge(m0_slice, on=["slice_name", "slice_value"], how="left")
slice_report["delta_vs_m0_slice_auc"] = slice_report["roc_auc"] - slice_report["m0_slice_auc"]
slice_report["slice_guard_triggered"] = (
    (slice_report["model_key"] != "m0_random_forest_frozen")
    & (slice_report["status"] == "computed")
    & (slice_report["m0_slice_status"] == "computed")
    & (slice_report["n"] >= MIN_SLICE_SIZE)
    & (slice_report["delta_vs_m0_slice_auc"] < -SLICE_DEGRADATION_THRESHOLD)
)

summary_df = pd.DataFrame(summary_rows)
summary_df["delta_vs_m0_oof"] = summary_df["oof_auc"] - m0_auc
summary_df["fold_deltas_vs_m0"] = ""
summary_df["same_sign_positive_folds"] = 0
summary_df["slice_guard_triggered"] = False
summary_df["classification"] = "not_classified"
summary_df["decision_reason"] = ""

for idx, row in summary_df.iterrows():
    model_key = row["model_key"]
    if row["run_status"] != "trained":
        summary_df.loc[idx, "classification"] = "failed_run"
        summary_df.loc[idx, "decision_reason"] = row.get("failure_reason", "Model failed to run.")
        continue
    model_fold = fold_metrics[fold_metrics["model_key"] == model_key].sort_values("fold")
    fold_deltas = model_fold["delta_vs_m0_fold_auc"].to_numpy()
    same_sign = int((fold_deltas > 0).sum())
    slice_guard = bool(slice_report[slice_report["model_key"] == model_key]["slice_guard_triggered"].any())
    summary_df.loc[idx, "fold_deltas_vs_m0"] = "|".join(f"{v:.12f}" for v in fold_deltas)
    summary_df.loc[idx, "same_sign_positive_folds"] = same_sign
    summary_df.loc[idx, "slice_guard_triggered"] = slice_guard
    if model_key == "m0_random_forest_frozen":
        summary_df.loc[idx, "classification"] = "reference_reproduced"
        summary_df.loc[idx, "decision_reason"] = f"M0 reproduced F2 within tolerance; max abs probability diff vs persisted F2 = {m0_max_abs_diff_vs_f2:.12g}."
    elif slice_guard and row["delta_vs_m0_oof"] >= OOF_GAIN_THRESHOLD and same_sign >= SAME_SIGN_FOLD_THRESHOLD:
        summary_df.loc[idx, "classification"] = "escalated"
        summary_df.loc[idx, "decision_reason"] = "OOF/fold evidence passed, but at least one mandatory slice degraded by more than 0.02 AUC vs m0."
    elif row["delta_vs_m0_oof"] >= OOF_GAIN_THRESHOLD and same_sign >= SAME_SIGN_FOLD_THRESHOLD and not slice_guard:
        summary_df.loc[idx, "classification"] = "promotable_evidence"
        summary_df.loc[idx, "decision_reason"] = "OOF gain, same-sign fold criterion, and slice guard all passed; project director selection still required."
    else:
        summary_df.loc[idx, "classification"] = "no_qualifying_evidence"
        summary_df.loc[idx, "decision_reason"] = "Did not meet the pre-registered OOF gain and/or same-sign fold evidence rule."

summary_cols = [
    "experiment_id",
    "model_key",
    "display_name",
    "run_status",
    "classification",
    "decision_reason",
    "model_family",
    "model_params_summary",
    "feature_set",
    "feature_count",
    "oof_auc",
    "delta_vs_m0_oof",
    "fold_mean_auc",
    "fold_std_auc",
    "fold_auc_scores",
    "fold_deltas_vs_m0",
    "same_sign_positive_folds",
    "slice_guard_triggered",
    "hpo_status",
    "submission_created",
    "leakage_warning",
]
summary_df = summary_df[summary_cols]
summary_df

,experiment_id,model_key,display_name,run_status,classification,decision_reason,model_family,model_params_summary,feature_set,feature_count,...,delta_vs_m0_oof,fold_mean_auc,fold_std_auc,fold_auc_scores,fold_deltas_vs_m0,same_sign_positive_folds,slice_guard_triggered,hpo_status,submission_created,leakage_warning
0,phase8_model_family_comparison_v1,m0_random_forest_frozen,RandomForest frozen Phase 7 F2 reference,trained,reference_reproduced,M0 reproduced F2 within tolerance; max abs pro...,RandomForestClassifier,"{""bootstrap"": true, ""ccp_alpha"": 0.0, ""class_w...",Phase 7 F2,21,...,0.000000,0.812456,0.029238,0.789388885748|0.832381561190|0.825854108957|0...,0.000000000000|0.000000000000|0.000000000000|0...,0,False,not_run,False,False
1,phase8_model_family_comparison_v1,m1_logistic_regression,LogisticRegression,trained,escalated,"OOF/fold evidence passed, but at least one man...",LogisticRegression,"{""C"": 1.0, ""class_weight"": null, ""dual"": false...",Phase 7 F2,21,...,0.015432,0.827682,0.029565,0.797529538131|0.850216634704|0.862816961432|0...,0.008140652383|0.017835073514|0.036962852475|0...,4,True,not_run,False,False
2,phase8_model_family_comparison_v1,m2_random_forest_default,RandomForest default depth,trained,no_qualifying_evidence,Did not meet the pre-registered OOF gain and/o...,RandomForestClassifier,"{""bootstrap"": true, ""ccp_alpha"": 0.0, ""class_w...",Phase 7 F2,21,...,-0.006126,0.806100,0.029678,0.790378201142|0.841579657646|0.822274309255|0...,0.000989315394|0.009198096456|-0.003579799702|...,2,True,not_run,False,False
3,phase8_model_family_comparison_v1,m3_extra_trees_default,ExtraTrees default,trained,no_qualifying_evidence,Did not meet the pre-registered OOF gain and/o...,ExtraTreesClassifier,"{""bootstrap"": false, ""ccp_alpha"": 0.0, ""class_...",Phase 7 F2,21,...,-0.021956,0.790354,0.030487,0.754621516197|0.826862703317|0.811826124014|0...,-0.034767369552|-0.005518857873|-0.01402798494...,0,True,not_run,False,False
4,phase8_model_family_comparison_v1,m4_hist_gradient_boosting,HistGradientBoosting,trained,no_qualifying_evidence,Did not meet the pre-registered OOF gain and/o...,HistGradientBoostingClassifier,"{""categorical_features"": ""from_dtype"", ""class_...",Phase 7 F2,21,...,-0.002321,0.809203,0.027170,0.798843914297|0.825257475673|0.818879181760|0...,0.009455028549|-0.007124085517|-0.006974927197...,1,True,not_run,False,False


## 8. Artifact Writing, Manifest, and Report

**Purpose.** Persist only the authorized Wave 1 artifacts with pre-write guards.

**Expected output.** OOF files, model summary, fold metrics, slice report, validation report, candidate log, and manifest are written.

**Leakage caution.** The main experiment log remains byte-identical; no submissions or acceptance record are created.

In [8]:
def write_csv_guarded(df: pd.DataFrame, path: Path) -> None:
    if path.exists():
        raise RuntimeError(f"Refusing to overwrite existing artifact: {path.relative_to(ROOT)}")
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def write_text_guarded(text: str, path: Path) -> None:
    if path.exists():
        raise RuntimeError(f"Refusing to overwrite existing artifact: {path.relative_to(ROOT)}")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8", newline="\n")


def markdown_table(df: pd.DataFrame, columns: list[str]) -> str:
    if df.empty:
        return "_No rows._"
    rendered = df[columns].copy()
    for col in rendered.columns:
        if pd.api.types.is_float_dtype(rendered[col]):
            rendered[col] = rendered[col].map(lambda x: "" if pd.isna(x) else f"{x:.12f}")
    rendered = rendered.astype(str)
    header = "| " + " | ".join(rendered.columns) + " |"
    separator = "| " + " | ".join(["---"] * len(rendered.columns)) + " |"
    lines = [header, separator]
    for _, row in rendered.iterrows():
        values = []
        for col in rendered.columns:
            value = str(row[col]).replace("\n", " ").replace("|", "\\|")
            values.append(value)
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines)


for model_key, oof_df in oof_by_model.items():
    write_csv_guarded(oof_df, OOF_PATHS[model_key])

write_csv_guarded(summary_df, PATHS["model_summary"])
write_csv_guarded(fold_metrics, PATHS["fold_metrics"])
write_csv_guarded(slice_report, PATHS["slice_report"])

candidate_log = summary_df.copy()
candidate_log.insert(0, "run_timestamp_utc", datetime.now(timezone.utc).isoformat())
candidate_log["notebook_or_script"] = "notebooks/08_phase8_model_family_comparison.ipynb"
candidate_log["git_commit_or_status"] = full_head
candidate_log["fold_strategy"] = "Frozen Phase 6 StratifiedKFold folds, sha16=96937649526bcadb"
candidate_log["random_seed"] = PROJECT_SEED
candidate_log["preprocessing_summary"] = "F2 shared fold-safe ColumnTransformer; median numeric imputation; role categorical most_frequent+OHE; LogisticRegression adds fold-fitted StandardScaler."
candidate_log["school_strategy"] = "School excluded from features; diagnostic slice only."
candidate_log["public_lb_used"] = False
write_csv_guarded(candidate_log, PATHS["candidate_log"])

report_lines = [
    "# Phase 8 Wave 1 Validation Report",
    "",
    "## Executive Summary",
    "",
    "Phase 8 Wave 1 executed the authorized sklearn-native model-family comparison on the accepted Phase 7/7B F2 feature set. This report is evidence only: candidate selection, acceptance, and commit remain gated on explicit project director authorization.",
    "",
    f"- Experiment ID: `{EXPERIMENT_ID}`",
    f"- Start HEAD: `{full_head}`",
    f"- Environment: Python {versions['python']}, pandas {versions['pandas']}, scikit-learn {versions['sklearn']}, numpy {versions['numpy']}",
    f"- M0 OOF ROC-AUC: `{m0_auc:.16f}`",
    f"- Persisted F2 OOF ROC-AUC: `{f2_auc:.16f}`",
    f"- M0 max absolute probability difference vs persisted F2: `{m0_max_abs_diff_vs_f2:.16g}`",
    "",
    "## Authorized Scope",
    "",
    "- Authorized models: m0, m1, m2, m3, m4.",
    "- Not authorized: m5, xgboost, lightgbm, catboost, deep tabular models, HPO, submissions, ensembles, leaderboard use, external data, package installation, environment mutation, Phase 9, Phase 10, Phase 11.",
    "- School remained excluded from all feature matrices.",
    "",
    "## Model Summary",
    "",
    markdown_table(summary_df, ["model_key", "run_status", "classification", "oof_auc", "delta_vs_m0_oof", "fold_mean_auc", "fold_std_auc", "same_sign_positive_folds", "slice_guard_triggered"]),
    "",
    "## Fold-Level Metrics",
    "",
    markdown_table(fold_metrics, ["model_key", "fold", "n_valid", "n_positive", "n_negative", "roc_auc", "delta_vs_m0_fold_auc"]),
    "",
    "## Slice Diagnostics Summary",
    "",
]
slice_guard_rows = slice_report[slice_report["slice_guard_triggered"]]
if slice_guard_rows.empty:
    report_lines.append("No mandatory slice degradation > 0.02 AUC versus m0 was detected for trained candidates.")
else:
    report_lines.append("The following mandatory slice guard rows triggered escalation:")
    report_lines.append("")
    report_lines.append(markdown_table(slice_guard_rows, ["model_key", "slice_name", "slice_value", "n", "n_positive", "n_negative", "roc_auc", "m0_slice_auc", "delta_vs_m0_slice_auc"]))
report_lines.extend(
    [
        "",
        "## Leakage Checklist",
        "",
        "- Official train/test/sample CSVs only; test used only for contract checks.",
        "- Frozen folds loaded from file and SHA-checked; folds were not recomputed.",
        "- F2 features are row-wise and fixed; no per-model feature engineering was introduced.",
        "- All learned preprocessing was fitted inside fold-specific sklearn pipelines.",
        "- Positive-class probabilities were extracted only after verifying `estimator.classes_` contains label `1` exactly once.",
        "- No target encoding, feature selection, dimensionality reduction, HPO, ensemble, submission, public leaderboard use, or external data was used.",
        "- `logs/experiment_log.csv` remained byte-identical; candidate log was written separately under `outputs/reports/`.",
        "",
        "## Threshold Interpretation",
        "",
        f"The OOF gain threshold `{OOF_GAIN_THRESHOLD}` and same-sign fold rule `{SAME_SIGN_FOLD_THRESHOLD}/5` are evidence flags, not an automatic winner-selection mechanism. The threshold was inherited from prior RF seed-noise analysis and may not fully describe per-family variance.",
        "",
        "## Phase Locks",
        "",
        "Phase 9 remains locked.",
        "Phase 10 remains locked.",
        "Phase 11 remains locked.",
    ]
)
write_text_guarded("\n".join(report_lines) + "\n", PATHS["validation_report"])

manifest_rows = []
for artifact_name, path in {
    **{f"oof_{key}": value for key, value in OOF_PATHS.items() if value.exists()},
    "model_summary": PATHS["model_summary"],
    "fold_metrics": PATHS["fold_metrics"],
    "slice_report": PATHS["slice_report"],
    "validation_report": PATHS["validation_report"],
    "candidate_log": PATHS["candidate_log"],
}.items():
    manifest_rows.append(
        {
            "experiment_id": EXPERIMENT_ID,
            "artifact_name": artifact_name,
            "path": str(path.relative_to(ROOT)),
            "exists": path.exists(),
            "sha256_16": hashlib.sha256(path.read_bytes()).hexdigest()[:16] if path.exists() else "",
            "size_bytes": path.stat().st_size if path.exists() else 0,
        }
    )
manifest_df = pd.DataFrame(manifest_rows)
write_csv_guarded(manifest_df, PATHS["artifact_manifest"])

if PATHS["main_log"].exists() and PATHS["main_log"].read_bytes() != main_log_before:
    raise RuntimeError("logs/experiment_log.csv changed during Phase 8 execution")

created_artifacts = [str(path.relative_to(ROOT)) for path in [*OOF_PATHS.values(), PATHS["model_summary"], PATHS["fold_metrics"], PATHS["slice_report"], PATHS["validation_report"], PATHS["candidate_log"], PATHS["artifact_manifest"]] if path.exists()]
created_artifacts

['outputs\\oof\\phase8_model_family_comparison_v1_m0_random_forest_frozen_oof_predictions.csv',
 'outputs\\oof\\phase8_model_family_comparison_v1_m1_logistic_regression_oof_predictions.csv',
 'outputs\\oof\\phase8_model_family_comparison_v1_m2_random_forest_default_oof_predictions.csv',
 'outputs\\oof\\phase8_model_family_comparison_v1_m3_extra_trees_default_oof_predictions.csv',
 'outputs\\oof\\phase8_model_family_comparison_v1_m4_hist_gradient_boosting_oof_predictions.csv',
 'outputs\\validation\\phase8_model_family_comparison_v1_model_summary.csv',
 'outputs\\validation\\phase8_model_family_comparison_v1_fold_metrics.csv',
 'outputs\\validation\\phase8_model_family_comparison_v1_slice_report.csv',
 'outputs\\reports\\phase8_model_family_comparison_v1_validation_report.md',
 'outputs\\reports\\phase8_model_family_comparison_v1_experiment_log_candidate.csv',
 'outputs\\reports\\phase8_model_family_comparison_v1_artifact_manifest.csv']

## 9. Independent In-Notebook Recompute

**Purpose.** Recompute persisted OOF metrics after writing and compare them against the saved summary.

**Expected output.** Differences are within `1e-9`; all persisted OOF files have valid probabilities and frozen fold alignment.

**Leakage caution.** This reads generated OOF artifacts only and does not train or select models.

In [9]:
persisted_summary = pd.read_csv(PATHS["model_summary"])
recompute_rows = []
for model_key in MODEL_KEYS:
    path = OOF_PATHS[model_key]
    if not path.exists():
        continue
    oof = pd.read_csv(path)
    if len(oof) != EXPECTED_TRAIN_ROWS:
        raise RuntimeError(f"{model_key}: persisted OOF row count invalid")
    if not oof["Id"].equals(train["Id"]) or not oof["fold"].equals(folds["fold"]):
        raise RuntimeError(f"{model_key}: persisted OOF alignment invalid")
    if not np.isfinite(oof["y_pred_proba"]).all() or not oof["y_pred_proba"].between(0, 1).all():
        raise RuntimeError(f"{model_key}: persisted OOF probabilities invalid")
    auc = float(roc_auc_score(oof["y_true"], oof["y_pred_proba"]))
    saved_auc = float(persisted_summary.loc[persisted_summary["model_key"] == model_key, "oof_auc"].iloc[0])
    diff = abs(auc - saved_auc)
    if diff > 1e-9:
        raise RuntimeError(f"{model_key}: persisted AUC recompute diff {diff} > 1e-9")
    recompute_rows.append({"model_key": model_key, "saved_auc": saved_auc, "recomputed_auc": auc, "abs_diff": diff})

if (ROOT / "docs" / "08_model_comparison" / "phase8_acceptance.md").exists():
    raise RuntimeError("phase8_acceptance.md was created, but this is not authorized")

post_forbidden_diff = git_stdout([
    "diff",
    "--name-only",
    "--",
    "data/input",
    "notebooks/_official",
    "references",
    "outputs/submissions",
    "logs/experiment_log.csv",
    ".vscode/settings.json",
])
if post_forbidden_diff:
    raise RuntimeError(f"Forbidden-path diff after run:\n{post_forbidden_diff}")

pd.DataFrame(recompute_rows)

,model_key,saved_auc,recomputed_auc,abs_diff
0,m0_random_forest_frozen,0.811650,0.811650,0.0
1,m1_logistic_regression,0.827082,0.827082,0.0
2,m2_random_forest_default,0.805524,0.805524,0.0
3,m3_extra_trees_default,0.789694,0.789694,0.0
4,m4_hist_gradient_boosting,0.809329,0.809329,0.0
